In [34]:
import math
from abc import ABC, abstractmethod
from pathlib import Path

import numpy as np
import requests

In [35]:
np.random.seed(42)

In [36]:
class Tensor:

    def __init__(self, data):
        self.data = np.array(data)
        self.grad = np.zeros_like(self.data)
        self.gradient_fn = None
        self.parents = set()

    def backward(self):
        topo = []
        visited = set()
        stack = [(self, False)]

        while stack:
            node, expanded = stack.pop()
            if node in visited:
                continue

            if expanded:
                visited.add(node)
                topo.append(node)
            else:
                stack.append((node, True))
                for p in node.parents:
                    if p not in visited:
                        stack.append((p, False))

        self.grad = np.ones_like(self.data)
        for t in reversed(topo):
            if t.gradient_fn is not None:
                t.gradient_fn()

        for t in topo:
            t.gradient_fn = lambda: None
            t.parents = set()

    def __add__(self, other):
        p = Tensor(self.data + other.data)

        def gradient_fn():
            self.grad += p.grad
            other.grad += p.grad

        p.gradient_fn = gradient_fn
        p.parents = {self, other}
        return p

    def __str__(self):
        return f'Tensor({self.data})'

In [37]:
class Dataset(ABC):

    def __init__(self, batch_size=1):
        self.batch_size = batch_size
        self.load()
        self.train()

    @abstractmethod
    def load(self):
        pass

    def train(self):
        self.data = self.train_data

    def eval(self):
        self.data = self.test_data

    def all(self):
        x, y = self.data
        return Tensor(x), Tensor(y)

    def __len__(self):
        return math.ceil(len(self.data[0]) / self.batch_size)

    def __getitem__(self, index):
        s = slice(index * self.batch_size, (index + 1) * self.batch_size)
        x, y = self.data
        return Tensor(x[s]), Tensor(y[s])

In [38]:
class CharDataset(Dataset):

    def __init__(self, filename, batch_size=1, context_size=64, stride=None, split=0.9):
        self.filename = filename
        self.context_size = context_size
        self.stride = stride if stride is not None else context_size // 2
        self.split = split
        super().__init__(batch_size)

    def load(self):
        with open(self.filename, encoding="utf-8") as f:
            text = f.read()

        self.vocab = sorted(set(text))
        self.vocab_size = len(self.vocab)
        self.stoi = {ch: i for i, ch in enumerate(self.vocab)}
        self.itos = {i: ch for i, ch in enumerate(self.vocab)}
        self.tokens = self.encode(text)

        split = int(len(self.tokens) * self.split)
        self.train_data = self._pack(self.tokens[:split])
        self.test_data = self._pack(self.tokens[split:])

    def _pack(self, tokens):
        onehot = np.eye(self.vocab_size)
        x, y = [], []
        for i in range(0, len(tokens) - self.context_size - 1, self.stride):
            x.append(tokens[i: i + self.context_size])
            y.append(onehot[tokens[i + 1: i + self.context_size + 1]])
        return x, y

    def encode(self, symbols):
        return [self.stoi[s] for s in symbols]

    def decode(self, tokens):
        return "".join(self.itos[t] for t in tokens)

In [39]:
class Layer(ABC):

    def __call__(self, *args):
        return self.forward(*args)

    @abstractmethod
    def forward(self, *args):
        pass

    @property
    def parameters(self):
        return []

In [40]:
class Linear(Layer):

    def __init__(self, in_size, out_size):
        self.weight = Tensor(np.random.randn(out_size, in_size) * np.sqrt(2 / in_size))
        self.bias = Tensor(np.random.rand(out_size))

    def forward(self, x: Tensor):
        p = Tensor(x.data @ self.weight.data.T + self.bias.data)

        def gradient_fn():
            self.weight.grad += p.grad.T @ x.data
            self.bias.grad += np.sum(p.grad, axis=0)
            x.grad += p.grad @ self.weight.data

        p.gradient_fn = gradient_fn
        p.parents = {x}
        return p

    @property
    def parameters(self):
        return [self.weight, self.bias]

In [41]:
class Composite(Layer, ABC):

    def __init__(self, layers):
        super().__init__()
        self.layers = list(layers)

    @property
    def parameters(self):
        return [p for l in self.layers for p in l.parameters]

In [42]:
class Embedding(Layer):

    def __init__(self, vocab_size, embedding_size, std=0.02):
        super().__init__()
        self.weight = Tensor(np.random.randn(vocab_size, embedding_size) * std)

    def forward(self, x: Tensor):
        p = Tensor(self.weight.data[x.data])

        def gradient_fn():
            np.add.at(self.weight.grad, x.data, p.grad)

        p.gradient_fn = gradient_fn
        p.parents = {self.weight}
        return p

    @property
    def parameters(self):
        return [self.weight]

In [43]:
class Tanh(Layer):

    def forward(self, x: Tensor):
        a = Tensor(np.tanh(x.data))

        def gradient_fn():
            x.grad += a.grad * (1 - a.data ** 2)

        a.gradient_fn = gradient_fn
        a.parents = {x}
        return a

In [44]:
class Loss(ABC):

    def __call__(self, p: Tensor, y: Tensor):
        return self.loss(p, y)

    @abstractmethod
    def loss(self, p: Tensor, y: Tensor):
        pass

In [45]:
class CELoss(Loss):

    def loss(self, p: Tensor, y: Tensor):
        exp = np.exp(p.data - np.max(p.data, axis=-1, keepdims=True))
        softmax = exp / np.sum(exp, axis=-1, keepdims=True)

        log = np.log(np.clip(softmax, 1e-10, 1))
        ce = Tensor(0 - np.sum(y.data * log) / len(y.data))

        def gradient_fn():
            p.grad += (softmax - y.data) / len(y.data)

        ce.gradient_fn = gradient_fn
        ce.parents = {p}
        return ce

In [46]:
class Optimizer(ABC):

    def __init__(self, parameters, lr):
        self.parameters = list(parameters)
        self.lr = lr

    @abstractmethod
    def step(self):
        pass

    def zero_grad(self):
        for p in self.parameters:
            p.grad = np.zeros_like(p.data)

In [47]:
class SGDOptimizer(Optimizer):

    def step(self):
        for p in self.parameters:
            p.data -= p.grad * self.lr

In [48]:
class AdamOptimizer(Optimizer):

    def __init__(self, parameters, lr=0.01, betas=(0.9, 0.999), eps=1e-8):
        super().__init__(parameters, lr)
        self.beta1, self.beta2 = betas
        self.eps = eps
        self.m: list[int | None] = [None] * len(parameters)
        self.v: list[int | None] = [None] * len(parameters)
        self.t = 0

    def step(self):
        self.t += 1
        for idx, p in enumerate(self.parameters):
            if p is not None:
                if self.m[idx] is None:
                    self.m[idx] = np.zeros_like(p.data)
                    self.v[idx] = np.zeros_like(p.data)

                self.m[idx] = self.beta1 * self.m[idx] + (1 - self.beta1) * p.grad
                self.v[idx] = self.beta2 * self.v[idx] + (1 - self.beta2) * (p.grad ** 2)
                m_hat = self.m[idx] / (1 - self.beta1 ** self.t)
                v_hat = self.v[idx] / (1 - self.beta2 ** self.t)
                self._apply_weight_decay(p)
                p.data -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

    def states(self):
        state = {"t": self.t}
        for idx, m in enumerate(self.m):
            if m is not None:
                state[f"m_{idx}"] = m
        for idx, v in enumerate(self.v):
            if v is not None:
                state[f"v_{idx}"] = v
        return state

    def load_states(self, state):
        self.t = int(state["t"])
        for idx in range(len(self.parameters)):
            if f"m_{idx}" in state:
                self.m[idx] = np.asarray(state[f"m_{idx}"])
                self.v[idx] = np.asarray(state[f"v_{idx}"])

    def _apply_weight_decay(self, p):
        pass

    def clip_grad_norm(self, max_norm=1.0):
        sq = 0.0
        for p in self.parameters:
            sq += float(np.sum(p.grad.astype(np.float64) ** 2))

        if np.sqrt(sq) > max_norm > 0:
            scale = max_norm / (np.sqrt(sq) + 1e-6)
            for p in self.parameters:
                p.grad *= scale

In [49]:
class AdamWOptimizer(AdamOptimizer):

    def __init__(self, parameters, lr=0.01, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01):
        super().__init__(parameters, lr, betas, eps)
        self.weight_decay = weight_decay

    def _apply_weight_decay(self, p):
        if p.data.ndim >= 2:
            p.data -= p.data * self.weight_decay * self.lr

In [50]:
class WarmupCosineScheduler:

    def __init__(self, max_lr, total_steps, warmup_steps, min_lr=0.0):
        self.max_lr = max_lr
        self.total_steps = max(total_steps, 1)
        self.warmup_steps = max(min(warmup_steps, self.total_steps), 0)
        self.min_lr = min_lr

    def step(self, current_step):
        if self.warmup_steps > 0 and current_step < self.warmup_steps:
            return self.max_lr * (current_step + 1) / self.warmup_steps

        if current_step >= self.total_steps:
            return self.min_lr

        progress = (current_step - self.warmup_steps) / max(self.total_steps - self.warmup_steps, 1)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return self.min_lr + (self.max_lr - self.min_lr) * cosine

In [51]:
class RNNCell(Composite):

    def __init__(self, in_size, out_size):
        self.input = Linear(in_size, out_size)
        self.hidden = Linear(out_size, out_size)
        self.tanh = Tanh()

        super().__init__([self.input,
                          self.hidden,
                          self.tanh])

    def forward(self, x: Tensor, h: Tensor):
        p = self.input(x)
        h = self.hidden(h)
        z = p + h

        def gradient_fn():
            p.grad += z.grad
            h.grad += z.grad

        z.gradient_fn = gradient_fn
        z.parents = {p, h}
        return self.tanh(z)

In [52]:
class RNN(Composite):

    def __init__(self, vocab_size, hidden_size, embedding_size):
        self.hidden_size = hidden_size

        self.embedding = Embedding(vocab_size, embedding_size)
        self.cell = RNNCell(embedding_size, hidden_size)
        self.output = Linear(hidden_size, vocab_size)

        super().__init__([self.embedding,
                          self.cell,
                          self.output])

    def step(self, token: Tensor, h: Tensor):
        p = self.embedding(token)
        h = self.cell(p, h)
        return self.output(h), h

    def forward(self, x: Tensor, h: Tensor = None):
        batch, seq_len = x.data.shape
        if h is None:
            h = Tensor(np.zeros((batch, self.hidden_size)))

        logits = []
        for t in range(seq_len):
            p, h = self.step(Tensor(x.data[:, t]), h)
            logits.append(p)

        return logits, h

In [53]:
class CNNModel:

    def __init__(self, layer, loss_fn, optimizer):
        self.layer = layer
        self.loss_fn = loss_fn
        self.optimizer = optimizer

    def train(self, dataset, epochs, scheduler=None,):
        dataset.train()

        steps = 0
        for epoch in range(epochs):
            for i in range(len(dataset)):
                if scheduler is not None:
                    self.optimizer.lr = scheduler.step(steps)

                feature, label = dataset[i]

                self.optimizer.zero_grad()
                prediction, _ = self.layer(feature)
                loss = Tensor(0.0)
                for t in range(len(prediction)):
                    loss += self.loss_fn(prediction[t], Tensor(label.data[:, t]))
                loss.backward()
                self.optimizer.step()
                steps += 1

    def evaluate(self, dataset):
        dataset.eval()

        feature, label = dataset.all()
        prediction, _ = self.layer(feature)
        loss = Tensor(0.0)
        for t in range(len(prediction)):
            loss += self.loss_fn(prediction[t], Tensor(label.data[:, t]))
        return prediction, loss

    def generate(self, dataset, prompt, steps=300):
        tokens = dataset.encode(prompt)
        h = Tensor(np.zeros((1, self.layer.hidden_size)))

        logits = None
        for token in tokens:
            logits, h = self.layer.step(Tensor([token]), h)

        for _ in range(steps):
            exp = np.exp(logits.data[0] - np.max(logits.data[0]))
            probs = exp / np.sum(exp)
            next_token = np.random.choice(len(probs), p=probs)
            tokens.append(next_token)
            logits, h = self.layer.step(Tensor([next_token]), h)

        return dataset.decode(tokens)

In [54]:
DATA_FILE = "tinyshakespeare.txt"

In [55]:
LEARNING_RATE = 0.002

In [56]:
BATCH_SIZE = 4

In [57]:
CONTEXT_SIZE = 32

In [58]:
HIDDEN_SIZE = 128

In [59]:
EMBEDDING_SIZE = 64

In [60]:
EPOCHS = 2

In [61]:
file = Path(DATA_FILE)
if not file.exists():
    file.parent.mkdir(parents=True, exist_ok=True)
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    response = requests.get(url)
    response.raise_for_status()
    file.write_text(response.text)

In [62]:
dataset = CharDataset(DATA_FILE, BATCH_SIZE, CONTEXT_SIZE)
layer = RNN(dataset.vocab_size, HIDDEN_SIZE, EMBEDDING_SIZE)
loss_fn = CELoss()
optimizer = AdamWOptimizer(layer.parameters, lr=LEARNING_RATE)
model = CNNModel(layer, loss_fn, optimizer)

In [63]:
scheduler = WarmupCosineScheduler(LEARNING_RATE, len(dataset), 100, LEARNING_RATE / 10)
model.train(dataset, EPOCHS)

In [64]:
prediction, loss = model.evaluate(dataset)

In [65]:
print(f'prediction: {len(prediction)} steps, each {prediction[0].data.shape}')
print(f'loss: {loss}')

prediction: 32 steps, each (6970, 65)
loss: Tensor(60.14989831141243)


In [66]:
print(model.generate(dataset, prompt="ROMEO:", steps=300))

ROMEO:
Is this dives well druscanh, my peet yee,.:
If state too knew the denice: ald Gromiouse, fet?
Them mest have the eise Fown, hed sefe, I all do?

KETHARIO:
I
And I newn do well you will and his frientite, I will withse. Bid Buhnior deliling you not.

KATHARINA:
I not you wrong? I knem seniency, I ne
